# SQLi Evaluation — Colab Wrapper

Thin wrapper around `evaluate.py`, `linear_probe.py`, `evaluate_baselines.py`,
and `compare.py`. All evaluation logic lives in those scripts; this notebook just:
1. Mounts Google Drive and installs dependencies
2. Clones the repo (or pulls latest)
3. Overrides key config values
4. Runs each evaluation script in sequence
5. Downloads results and charts

**Evaluation order (everyone sits the same exam — the test split):**

| Step | Script | Models | Answers |
|---|---|---|---|
| 1 | `evaluate.py` | codebert-sqli, bert-sqli-500k, codebert-sqli-500k | PP1, PP6 (after course) |
| 2 | `linear_probe.py` | bert-base-probe, codebert-base-probe | PP6 (before course) |
| 3 | `evaluate_baselines.py` | LR, RF, SVM | PP5 |
| 4 | `evaluate_ollama.py` *(local)* | Llama 3.2 3B | PP4 |
| 5 | `compare.py` | — | tables + charts |

**Before running:**
- Set runtime to GPU: Runtime → Change runtime type → Hardware accelerator → **GPU (T4)**
- Upload `compiled_dataset.csv` to `My Drive/mestrado/compiled_dataset.csv`
- Train the three models with `train_colab.ipynb` first (one run per `RUN` value) —
  their folders must exist under `My Drive/mestrado/models/`

**Note:** `evaluate_ollama.py` (PP4 — Llama 3.2 3B) requires a local Ollama server
and **cannot run on Colab**. See the note cell below.

In [ ]:
# ── Cell 1: Environment setup ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Colab already ships a CUDA-enabled torch — do NOT include torch here.
# pip install torch pulls the CPU-only wheel and overwrites Colab's CUDA build.
!pip install -q pandas scikit-learn imbalanced-learn transformers matplotlib seaborn psutil codecarbon

import os
import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

REPO_DIR = '/content/mestrado'
if not os.path.exists(REPO_DIR):
    !git clone -b eval-comparison https://github.com/l-a-motta/mestrado-sqli-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── Cell 2: Flush GPU and RAM before evaluation ───────────────────────────────
import gc
import torch
import psutil

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    props = torch.cuda.get_device_properties(0)
    free  = props.total_memory - torch.cuda.memory_allocated(0)
    print(f'GPU  : {props.name}  total={props.total_memory/1e9:.1f} GB  free={free/1e9:.1f} GB')

ram = psutil.virtual_memory()
print(f'RAM  : total={ram.total/1e9:.1f} GB  available={ram.available/1e9:.1f} GB')

In [ ]:
# ── Cell 3: evaluate.py — configuration overrides ────────────────────────────
import sys
from pathlib import Path

sys.path.insert(0, f'{REPO_DIR}/code/eval')
sys.path.insert(0, f'{REPO_DIR}/code/fine-tuning')

import evaluate as e

DRIVE_DATASET    = '/content/drive/MyDrive/mestrado/compiled_dataset.csv'
DRIVE_MODELS_DIR = '/content/drive/MyDrive/mestrado/models'
DRIVE_RESULTS    = Path('/content/drive/MyDrive/mestrado/eval_results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

e.DATASET_PATH = DRIVE_DATASET
e.RESULTS_DIR  = DRIVE_RESULTS
e.BATCH_SIZE   = 64      # T4 (16 GB): 64 is safe for inference; try 128 if no OOM

# The three fine-tuned models (D3.7) — trained with train_colab.ipynb, one per RUN
e.MODELS = [
    ('codebert-sqli',      f'{DRIVE_MODELS_DIR}/codebert-sqli'),       # product (full corpus)
    ('bert-sqli-500k',     f'{DRIVE_MODELS_DIR}/bert-sqli-500k'),      # PP6 control (capped)
    ('codebert-sqli-500k', f'{DRIVE_MODELS_DIR}/codebert-sqli-500k'),  # PP6 control (capped)
]

print(f'Dataset   : {e.DATASET_PATH}')
print(f'Results   : {e.RESULTS_DIR}')
print(f'Batch     : {e.BATCH_SIZE}')
for mid, mpath in e.MODELS:
    print(f'  {mid:20s} → {mpath}')

In [ ]:
# ── Cell 4: Run evaluate.py (BERT models — expect ~1–3 h on T4 for full split) ─
e.main()

In [ ]:
# ── Cell 4b: linear_probe.py — configuration overrides (PP6 untuned controls) ─
import linear_probe as lp

lp.DATASET_PATH = DRIVE_DATASET
lp.RESULTS_DIR  = DRIVE_RESULTS
lp.BATCH_SIZE   = 64

print(f'Dataset         : {lp.DATASET_PATH}')
print(f'Results         : {lp.RESULTS_DIR}')
print(f'Probe train rows: {lp.PROBE_TRAIN_ROWS:,}')
for mid, mpath in lp.ENCODERS:
    print(f'  {mid:20s} → {mpath}')

In [ ]:
# ── Cell 4c: Run linear_probe.py (frozen encoders + LR — expect ~1–2 h on T4) ─
lp.main()

In [ ]:
# ── Cell 5: evaluate_baselines.py — configuration overrides ──────────────────
import evaluate_baselines as b

b.DATASET_PATH   = DRIVE_DATASET
b.RESULTS_DIR    = DRIVE_RESULTS
b.MAX_TRAIN_ROWS = 100_000   # match Stage A BERT training size for fair PP5 comparison

print(f'Dataset       : {b.DATASET_PATH}')
print(f'Results       : {b.RESULTS_DIR}')
print(f'Max train rows: {b.MAX_TRAIN_ROWS:,}')

In [ ]:
# ── Cell 6: Run evaluate_baselines.py (traditional ML — expect ~5–15 min CPU) ─
b.main()

## evaluate_ollama.py — run locally (PP4, not on Colab)

`evaluate_ollama.py` requires a local Ollama server and **cannot run on Colab**.

Run from the repo root on your local machine:
```bash
# 1. Install Ollama: https://ollama.com
# 2. ollama pull llama3.2:3b
# 3. python code/eval/evaluate_ollama.py
```

After it finishes, copy `pp4_*.csv` and the updated `evaluation_results.csv` from
`code/eval/results/` into `My Drive/mestrado/eval_results/` before running
`compare.py` below, so PP4 is included in the final charts.

In [ ]:
# ── Cell 7: compare.py — configuration overrides ─────────────────────────────
# Run after evaluate_ollama.py has completed locally and its results have been
# uploaded to DRIVE_RESULTS (see note above). PP4 charts are silently skipped
# if pp4 rows are absent from evaluation_results.csv.

sys.path.insert(0, f'{REPO_DIR}/code/comparison')
import compare as c

DRIVE_COMPARISON = Path('/content/drive/MyDrive/mestrado/comparison_output')
DRIVE_COMPARISON.mkdir(parents=True, exist_ok=True)

c.RESULTS_DIR = DRIVE_RESULTS
c.OUTPUT_DIR  = DRIVE_COMPARISON

print(f'Results dir  : {c.RESULTS_DIR}')
print(f'Output dir   : {c.OUTPUT_DIR}')

In [ ]:
# ── Cell 8: Run compare.py (chart generation — fast, ~1 min) ─────────────────
c.main()

In [ ]:
# ── Cell 9: Download results and charts to local machine ─────────────────────
import glob
from google.colab import files

# Summary metrics table
summary_csv = str(DRIVE_RESULTS / 'evaluation_results.csv')
if os.path.exists(summary_csv):
    files.download(summary_csv)
    print(f'Downloaded: {summary_csv}')
else:
    print('evaluation_results.csv not found — run the eval cells first.')

# Per-row prediction CSVs
for pred in sorted(glob.glob(str(DRIVE_RESULTS / '*_predictions.csv'))):
    files.download(pred)
    print(f'Downloaded: {pred}')

# Comparison charts and tables
chart_files = (
    sorted(glob.glob(str(DRIVE_COMPARISON / '*.png'))) +
    sorted(glob.glob(str(DRIVE_COMPARISON / '*.csv')))
)
for f in chart_files:
    files.download(f)
    print(f'Downloaded: {f}')

# Latest eval log files
for log in sorted(glob.glob('code/eval/logs/evaluate_*.log'))[-3:]:
    files.download(log)
    print(f'Downloaded: {log}')